# SASRec on recent movies (MovieLens 32M, cut to 1M ratings)

The same SASRec notebook as `notebooks/movielens/sasrec.ipynb`, on the dataset made by `01_prep.ipynb`: 4,000
movies released 2010-2023, 10,628 users, ~1M ratings. Random and most-popular are added for context.

**SASRec** (Kang & McAuley, 2018, "Self-Attentive Sequential Recommendation") is the standard sequential
recommender: a small transformer that reads a user's rating history (movie IDs only - no titles, plots or genres)
and predicts the next movie (`genrec.models.sasrec`). It is the usual yardstick for models like the fine-tuned Qwen.

Same split and test as every other model: train on `items[:-2]`, validate on `items[-2]`, test on `items[-1]`; the
same 1,000 test users; full ranking over the catalogue, already-rated movies skipped; hit@1, HR@10, NDCG@10.

Two variants: **SASRec-200** (the standard setup, up to 200 past ratings) and **SASRec-20** (only the last 20, the
same information the Qwen prompt gets). Trains in ~15 minutes on a laptop; the models are saved to `artifacts/`
for the serving endpoint.

Note on numbers in papers: the SASRec paper reports Hit@10 = 0.82 on ML-1M because it ranks the true movie among
only 100 sampled negatives; here every movie in the catalogue is a candidate (full ranking), which is much harder
and the more reliable protocol (Krichene & Rendle, 2020).

In [1]:
import pandas as pd
import torch

from genrec.data import Dataset, history, sample_test_users, target
from genrec.metrics import evaluate_scores
from genrec.models import baselines as B
from genrec.models import sasrec
from genrec.paths import ARTIFACTS_DIR, DATA_DIR, RESULTS_DIR

DATA = DATA_DIR / "ml-32m-recent"
SAVE_DIR = ARTIFACTS_DIR / "ml32m-recent"

DEVICE = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
VARIANTS = {"SASRec-200": sasrec.SASRecConfig(max_len=200), "SASRec-20": sasrec.SASRecConfig(max_len=20)}
print("device:", DEVICE)

device: mps


## 1. Data

In [2]:
ds = Dataset.load(DATA)
seqs = ds.encoded()
test_users = sample_test_users(ds.users)
title = dict(zip(range(1, ds.index.n_cols), ds.movies.set_index("MovieID").loc[ds.index.movie_ids, "Title"]))
lengths = pd.Series([len(s) for s in seqs.values()])
print(f"{len(seqs)} users, {ds.index.n_cols - 1} movies, {len(ds.ratings)} ratings; "
      f"ratings per user: median {lengths.median():.0f}, max {lengths.max()}")

10628 users, 4000 movies, 999878 ratings; ratings per user: median 53, max 1702


## 2. Train both variants

Every epoch: a random window of each user's training history, every position predicted (full softmax). The epoch
with the best validation NDCG@10 is kept.

In [3]:
models = {}
for name, cfg in VARIANTS.items():
    print(f"--- {name}")
    models[name], _ = sasrec.train(seqs, ds.index.n_cols - 1, cfg, DEVICE, log=lambda m, n=name: print(f"{n} {m}"))
    sasrec.save(models[name], ds.index.movie_ids, SAVE_DIR / name.lower())
print("saved to", SAVE_DIR)

--- SASRec-200


SASRec-200 epoch   1  loss   600.9  val HR@10 0.084  NDCG@10 0.043  (best 0.043 @ 1)  13s


SASRec-200 epoch  10  loss   507.2  val HR@10 0.174  NDCG@10 0.090  (best 0.090 @ 10)  110s


SASRec-200 epoch  20  loss   491.5  val HR@10 0.193  NDCG@10 0.102  (best 0.102 @ 20)  204s


SASRec-200 epoch  30  loss   486.9  val HR@10 0.205  NDCG@10 0.109  (best 0.109 @ 30)  299s


...

SASRec-20 epoch  40  loss   476.1  val HR@10 0.184  NDCG@10 0.097  (best 0.100 @ 35)  41s


SASRec-20 epoch  50  loss   474.5  val HR@10 0.191  NDCG@10 0.101  (best 0.103 @ 48)  52s


SASRec-20 epoch  60  loss   472.4  val HR@10 0.195  NDCG@10 0.104  (best 0.105 @ 57)  62s


SASRec-20 epoch  70  loss   470.2  val HR@10 0.195  NDCG@10 0.104  (best 0.105 @ 57)  72s


SASRec-20 stopped at epoch 77, best validation NDCG@10 0.105 at epoch 57, 79s, 0.31M params
saved to artifacts/ml32m-recent


## 3. Test-set comparison

In [4]:
def test_metrics(model, us):
    return sasrec.evaluate(model, [history(seqs[u], "test") for u in us], [target(seqs[u], "test") for u in us], DEVICE)



def baselines(us):
    """Random and most-popular scores for these users - the floor, for context."""
    hist = [history(seqs[u], "test") for u in us]
    X = B.interaction_matrix([history(seqs[u], "test") for u in ds.users], ds.index.n_cols)
    return hist, {"random": B.random_scores(len(us), ds.index.n_cols), "most popular": B.popularity_scores(X, len(us))}

results = {name: test_metrics(m, test_users) for name, m in models.items()}
base_hist, base_scores = baselines(test_users)
results |= {name: evaluate_scores(s, base_hist, [target(seqs[u], "test") for u in test_users])
            for name, s in base_scores.items()}
table = pd.DataFrame(results).T[["hit@1", "HR@10", "NDCG@10"]]
RESULTS_DIR.mkdir(exist_ok=True)
table.to_csv(RESULTS_DIR / "ml32m_recent_sasrec_test.csv")
print(f"test: last rating of {len(test_users)} users")
table.style.format("{:.3f}")

test: last rating of 1000 users


,hit@1,HR@10,NDCG@10
SASRec-200,0.042,0.182,0.100
SASRec-20,0.040,0.174,0.097
random,0.000,0.001,0.000
most popular,0.018,0.088,0.046


### All users

In [5]:
pd.DataFrame({name: test_metrics(m, ds.users) for name, m in models.items()}).T.style.format("{:.3f}")

,hit@1,HR@10,NDCG@10
SASRec-200,0.044,0.202,0.110
SASRec-20,0.037,0.184,0.099


## 4. Example recommendations (SASRec-200)

In [6]:
for u in test_users[:5]:
    hist, tgt = history(seqs[u], "test"), target(seqs[u], "test")
    top = sasrec.recommend(models["SASRec-200"], hist, DEVICE)
    verdict = f"HIT at rank {top.index(tgt) + 1}" if tgt in top else "miss"
    print(f"user {u}  last rated: {'; '.join(title[m] for m in hist[-5:])}")
    print(f"   real next: {title[tgt]}   {verdict}")
    print(f"   top 10:    {'; '.join(title[m] for m in top)}\n")

user 119156  last rated: The Martian (2015); Deadpool (2016); Star Wars: Episode VII - The Force Awakens (2015); Captain America: The Winter Soldier (2014); John Wick (2014)
   real next: Spotlight (2015)   miss
   top 10:    Captain America: Civil War (2016); Avengers: Age of Ultron (2015); Ant-Man (2015); Doctor Strange (2016); Kingsman: The Secret Service (2015); Rogue One: A Star Wars Story (2016); X-Men: Days of Future Past (2014); Mission: Impossible - Rogue Nation (2015); Star Trek Into Darkness (2013); Guardians of the Galaxy 2 (2017)

user 130513  last rated: Safety Not Guaranteed (2012); Inception (2010); The Imitation Game (2014); Shutter Island (2010); Room (2015)
   real next: Misérables, Les (2012)   miss
   top 10:    The Martian (2015); Big Short, The (2015); Inside Out (2015); Ex Machina (2015); Deadpool (2016); Spotlight (2015); Gone Girl (2014); Me and Earl and the Dying Girl (2015); Arrival (2016); Interstellar (2014)

user 13038  last rated: Tenet (2020); Black Swa